# Lab: Simon's Algorithm with Qiskit

In this lab we implement **Simon's algorithm**, a quantum algorithm that solves a promise problem exponentially faster than any known classical algorithm.

We are given a black-box ("oracle") function

$$f: \{0, 1\}^n \rightarrow \{0, 1\}^n$$

with the **promise** that there is a secret bitstring $s \in \{0, 1\}^n$, with $s \neq 0$, such that for all $x, y \in \{0, 1\}^n$,

$$f(x) = f(y) \iff y = x \text{ or } y = x \oplus s.$$

In other words, $f$ is **two-to-one** and pairs up inputs that differ by XOR with the secret: $f(x) = f(x \oplus s)$. Our job is to recover $s$.

- A classical algorithm needs $\Omega(2^{n/2})$ oracle queries in the worst case.
- Simon's algorithm recovers $s$ with $O(n)$ oracle queries.

We will build the algorithm in three tasks:
1. Implement the oracle $U_f$ for a given secret $s$.
2. Implement the circuit that runs the algorithm.
3. Implement the algorithm that runs the circuit and returns the secret string $s$.


In [ ]:
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator

print("Qiskit imported successfully!")


### Helper Functions & Guardrails

The oracle acts on $n$ input qubits **plus $n$ output qubits**, for a total of $2n$ qubits. We adopt the convention that the **input register** is qubits $0$ through $n-1$ and the **output register** is qubits $n$ through $2n-1$.

The helpers below are used by the test cells:

- `validate_secret(s)` checks that `s` is a non-empty, non-zero binary string.
- `validate_oracle_inputs(n, oracle)` checks that `n` is valid and the oracle acts on `2n` qubits.
- `all_basis_states(n)` returns every $n$-bit input as a bitstring.
- `oracle_output(oracle, n, x)` evaluates the oracle classically by simulating $U_f |x\rangle|0^n\rangle$ and reading the output register.
- `nullspace_mod2(rows, n)` performs the classical post-processing: given measured bitstrings $y$ as rows, it returns a basis, over GF(2), for the set of all secrets $s$ satisfying $y \cdot s = 0 \pmod 2$ for every row.


In [ ]:
def validate_secret(s: str) -> None:
    """Guardrail: checks s is a non-empty, non-zero binary string."""
    assert isinstance(s, str), "The secret must be a string."
    assert len(s) >= 1, "The secret must have at least one bit."
    assert set(s).issubset({'0', '1'}), (
        f"Secret '{s}' must only contain '0' and '1'."
    )
    assert '1' in s, "The secret must be non-zero (Simon's problem assumes s != 0)."


def validate_oracle_inputs(n: int, oracle: QuantumCircuit | None = None) -> None:
    """Guardrail: checks n is a positive integer and the oracle has 2n qubits."""
    assert isinstance(n, int) and n >= 1, "n must be a positive integer."
    if oracle is not None:
        assert oracle.num_qubits == 2 * n, (
            f"Oracle must act on 2n = {2 * n} qubits "
            f"(n inputs plus n outputs), but it acts on {oracle.num_qubits}."
        )


def all_basis_states(n: int) -> list[str]:
    """Returns every n-bit computational basis state as a bitstring."""
    return [format(i, f'0{n}b') for i in range(2 ** n)]


def oracle_output(oracle: QuantumCircuit, n: int, x: str) -> str:
    """
    Evaluates the oracle f on the input x.

    Simulates U_f |x>|0^n>, where the output register is qubits n..2n-1,
    and returns f(x) as an n-bit bitstring.
    """
    validate_oracle_inputs(n, oracle)
    assert len(x) == n, f"Input '{x}' must have length n = {n}."
    assert set(x).issubset({'0', '1'}), f"Input '{x}' must be a binary string."

    circuit = QuantumCircuit(2 * n)
    for i, bit in enumerate(reversed(x)):
        if bit == '1':
            circuit.x(i)
    circuit.compose(oracle, inplace=True)

    statevector = Statevector(circuit)
    probabilities = statevector.probabilities(range(n, 2 * n))
    index = max(range(len(probabilities)), key=lambda i: probabilities[i])
    return format(index, f'0{n}b')


def nullspace_mod2(rows: list[int], n: int) -> list[int]:
    """
    Returns a basis for the nullspace over GF(2) of the given rows.

    Each row is an integer bitmask representing a constraint y . s = 0 mod 2.
    The returned list of integer bitmasks spans every solution s.
    """
    rows = list({row for row in rows if row})
    pivots = {}
    r = 0
    for col in range(n):
        pivot = next(
            (i for i in range(r, len(rows)) if (rows[i] >> col) & 1), None
        )
        if pivot is None:
            continue
        rows[r], rows[pivot] = rows[pivot], rows[r]
        for i in range(len(rows)):
            if i != r and (rows[i] >> col) & 1:
                rows[i] ^= rows[r]
        pivots[col] = r
        r += 1
        if r == len(rows):
            break

    free_columns = [col for col in range(n) if col not in pivots]
    basis = []
    for free in free_columns:
        vector = 1 << free
        for col, row in pivots.items():
            if (rows[row] >> free) & 1:
                vector |= 1 << col
        basis.append(vector)
    return basis


print("Helper functions and guardrails loaded successfully!")


## Task 1: Building the Oracle

An oracle is a reversible circuit $U_f$ that acts on $|x\rangle|y\rangle$ as

$$U_f |x\rangle |y\rangle = |x\rangle |y \oplus f(x)\rangle.$$

Here $x, y \in \{0, 1\}^n$, so the circuit acts on $2n$ qubits: the input register holds $x$ and the output register holds $y$.

For Simon's problem the oracle must encode a two-to-one function $f$ with hidden period $s$. A convenient construction is the linear map

$$f(x)_j = x_j \oplus s_j \, x_k,$$

where $k$ is any qubit index with $s_k = 1$ (for example, the lowest such index).

This function has exactly the properties we need:

- $f(x \oplus s)_j = (x_j \oplus s_j) \oplus s_j (x_k \oplus s_k) = f(x)_j$, since $s_k = 1$ and $s_j^2 = s_j$ over $\{0,1\}$.
- $f(x) = 0$ forces $x_j = s_j x_k$ for every $j$. If $x_k = 0$ then $x = 0$; if $x_k = 1$ then $x = s$. So the kernel is exactly $\{0, s\}$, making $f$ two-to-one.

In gate terms, we copy each input qubit into the output register with a CNOT, and for every set bit of $s$ we add $x_k$ to that output qubit with another CNOT.


### Task 1.1: Implement the Oracle

Implement `simon_oracle(circuit: QuantumCircuit, s: str) -> QuantumCircuit`.

- `circuit`: a `2n`-qubit circuit, where $n = $ `len(s)`; the input register is qubits $0 \dots n-1$ and the output register is qubits $n \dots 2n-1$.
- `s`: the secret bitstring of length $n$.
- Returns the same circuit with the oracle gates applied, implementing a two-to-one function with hidden period $s$.

> Hint: in a bitstring `s_{n-1} ... s_1 s_0`, the character `s[-1-i]` corresponds to qubit `i` (Qiskit uses little-endian ordering). A CNOT with control `i` and target `n + i` copies qubit `i` into the output register. To build $f(x)_j = x_j \oplus s_j x_k$, also apply a CNOT from qubit `k` to qubit `n + j` for every `j` where `s_j = 1`.


In [ ]:
def simon_oracle(circuit: QuantumCircuit, s: str) -> QuantumCircuit:
    """
    Implements an oracle f with hidden period s on the given circuit.

    The circuit has 2n qubits: the input register is qubits 0..n-1 and the
    output register is qubits n..2n-1. The oracle maps |x>|y> to
    |x>|y xor f(x)>.

    Args:
        circuit: A 2n-qubit circuit, where n = len(s).
        s: The secret bitstring of length n.

    Returns:
        QuantumCircuit: The same circuit with the oracle gates applied.
    """
    validate_secret(s)
    validate_oracle_inputs(len(s), circuit)

    # TODO: apply the oracle gates to the given circuit
    pass


### Task 1.2: Test the Oracle

The cell below checks that your oracle implements a valid Simon function for several secrets: every output is hit exactly **twice**, and $f(x) = f(x \oplus s)$ for every input $x$.


In [ ]:
from collections import Counter

for s in ["1", "10", "11", "101", "110", "011", "1001", "1010", "1111"]:
    n = len(s)
    oracle = simon_oracle(QuantumCircuit(2 * n), s)
    outputs = {x: oracle_output(oracle, n, x) for x in all_basis_states(n)}

    counts = Counter(outputs.values())
    assert all(count == 2 for count in counts.values()), (
        f"Oracle for s={s} is not two-to-one: {dict(counts)}"
    )

    for x, value in outputs.items():
        paired = format(int(x, 2) ^ int(s, 2), f'0{n}b')
        assert outputs[paired] == value, (
            f"Oracle for s={s} violates f(x) = f(x xor s) at x={x}"
        )

print("Oracle passed all tests!")


## Task 2: Building the Algorithm Circuit

Implement `simon_circuit(n: int, oracle: QuantumCircuit) -> QuantumCircuit`.

- `n`: number of input qubits.
- `oracle`: a `2n`-qubit oracle circuit, as built above.
- Returns a circuit implementing one round of Simon's algorithm, with `n` classical bits measuring the input register.

```
                      ┌───────┐
q_0:      |0> ─ H ─ ─ ┤       ├─ H ─ M
q_1:      |0> ─ H ─ ─ ┤       ├─ H ─ M
  ⋮              ⋮     │  U_f  │  ⋮   ⋮
q_(n-1):  |0> ─ H ─ ─ ┤       ├─ H ─ M
q_n:      |0> ─ ─ ─ ─ ┤       ├─
  ⋮              ⋮     │       │
q_(2n-1): |0> ─ ─ ─ ─ ┤       ├─
                      └───────┘
```

> Why it works: after the first layer of Hadamards the input register holds a superposition of all $x$. The oracle entangles it with the output register, so the input register collapses to a superposition of the pair $|x\rangle + |x \oplus s\rangle$. The final Hadamards ensure that every measured bitstring $y$ satisfies $y \cdot s = 0 \pmod 2$. Repeating the circuit collects enough such equations to solve for $s$.


In [ ]:
def simon_circuit(n: int, oracle: QuantumCircuit) -> QuantumCircuit:
    """
    Builds a circuit implementing one round of Simon's algorithm.

    Args:
        n: Number of input qubits.
        oracle: A 2n-qubit oracle circuit.

    Returns:
        QuantumCircuit: A circuit with n classical bits measuring the input
        register.
    """
    validate_oracle_inputs(n, oracle)

    # TODO: implement the circuit shown in the diagram above
    pass


## Task 3: Running the Algorithm

Implement `simon_algorithm(n: int, oracle: QuantumCircuit) -> str`.

- `n`: number of input qubits.
- `oracle`: a `2n`-qubit oracle circuit.
- Returns the secret bitstring `s` of length `n`.

Each run of the circuit returns a bitstring $y$ that is orthogonal to the secret: $y \cdot s = 0 \pmod 2$. A single run is not enough, so run the circuit repeatedly and collect distinct measurement outcomes until the equations pin down $s$ uniquely (that is, until the nullspace has dimension 1).

> Hint: build the circuit with `simon_circuit`, run it on `AerSimulator`, convert each measured bitstring to an integer with `int(bitstring, 2)`, and pass the collected rows to `nullspace_mod2(rows, n)`. When the returned basis has exactly one vector, convert it back to a bitstring with `format(basis[0], f'0{n}b')`.


In [ ]:
def simon_algorithm(n: int, oracle: QuantumCircuit) -> str:
    """
    Runs Simon's algorithm and returns the hidden secret s.

    Args:
        n: Number of input qubits.
        oracle: A 2n-qubit oracle circuit.

    Returns:
        str: The recovered secret bitstring of length n.
    """
    validate_oracle_inputs(n, oracle)

    # TODO: build the circuit with simon_circuit, run it on AerSimulator,
    # collect measured bitstrings, and use nullspace_mod2 to recover the secret.
    pass


### Final Test

The cell below recovers a variety of secrets using your implementation. It must pass for the lab to be complete.


In [ ]:
for s in ["1", "10", "11", "101", "110", "011", "1001", "1010", "1111", "10110"]:
    n = len(s)
    oracle = simon_oracle(QuantumCircuit(2 * n), s)
    recovered = simon_algorithm(n, oracle)
    assert recovered == s, (
        f"Failed to recover secret s={s}: got {recovered}"
    )

print("Simon's algorithm passed all tests!")
